# KL Expansion Samples from the Inverse Laplacian Prior

This notebook visualizes samples from a zero-mean Gaussian random field whose covariance operator is the inverse of the negative Laplacian $(-\Delta)^{-1}$ on a 3D box domain, discretized via a truncated tensor-product spectral expansion (KL expansion).

**Mathematical Context:**
- The prior is a Gaussian measure $\mathcal{N}(0, C)$ where $C = (-\Delta)^{-1}$.
- Samples are draws from this measure: $u = \sum \sqrt{\lambda_k} \xi_k \phi_k$, where $\xi_k \sim \mathcal{N}(0,1)$.
- Lower-order modes (larger eigenvalues of $C$) contribute more variance, meaning the prior favors smooth functions.

In [ ]:
import time

import numpy as np

from pygeoinf3d import BoxDomain, Lebesgue3D
from pygeoinf3d.core.functions import Function3D
from pygeoinf3d.operators.inverse_laplacian_box import InverseBoxLaplacian
from pygeoinf3d.plot import (
    compare_function3d_axis_sliders,
    evaluate_function3d_on_regular_grid,
    plot_function3d_axis_slider,
    plot_function3d_isosurface,
    plot_function3d_volume_dropdown,
)
from pygeoinf3d.providers.tensor_product import TensorProductProvider
from pygeoinf3d.sampling import KLSampler3D

In [ ]:
# Domain setup
box = BoxDomain(np.zeros(3), np.ones(3))
space = Lebesgue3D(box)

# Spectral discretization: 10x10x10 Dirichlet modes
provider = TensorProductProvider(box, truncation=(20, 20, 20), bc_type='dirichlet')
print(f"Total modes available: {provider.dim}")

# Covariance operator: inverse negative Laplacian
inv_lap = InverseBoxLaplacian(space, provider)
print(f"Eigenvalue range: {inv_lap.get_eigenvalue(0):.4f} ... {inv_lap.get_eigenvalue(provider.dim-1):.6f}")

# KL sampler
rng = np.random.default_rng(42)
kl = KLSampler3D(inv_lap, n_modes=8000, rng=rng)

In [ ]:
N_SAMPLES = 5
sample_labels = [f"KL Sample {i + 1}" for i in range(N_SAMPLES)]

print(f"Drawing {N_SAMPLES} prior samples from the KL approximation...")
samples_fn = kl.samples(N_SAMPLES)
print("Generated samples:")
for label in sample_labels:
    print(f"  - {label}")


### Visualization Approach

The plot helpers below handle grid evaluation internally. Because the sampler uses a , every sample is automatically tagged with a vectorized tensor-product evaluator — no code change is needed to get the speedup.

** knob:** Pass  to any plot helper to cap the number of KL modes evaluated during visualization. The underlying sample still carries all  — this only affects rendering speed vs. visual detail.

In [ ]:
plot_function3d_volume_dropdown(
    samples_fn,
    resolution=25,
    labels=sample_labels,
    colorscale="RdBu_r",
    robust_percentile=95.0,
    show_plot=False,
)

In [ ]:
plot_function3d_isosurface(
    samples_fn[0],
    resolution=30,
    n_levels=5,
    sigma=1.0,
    opacity=0.45,
    colorscale="RdBu_r",
    title="KL Sample 1 \u2014 isosurfaces (\u00b11\u03c3)",
    show_plot=False,
)

### Notes on the Prior

- With **Dirichlet boundary conditions**, all samples vanish at the boundary.
- The operator eigenvalues $\lambda_k = 1/(\pi^2(n_x^2/L_x^2 + n_y^2/L_y^2 + n_z^2/L_z^2))$ decay as the mode number increases.
- More KL modes $\to$ more fine-scale variability in the samples.
- The smoothness of samples from $(-\Delta)^{-1}$ is controlled by the spectral decay rate.

## Axis-Slice Sliders

These views use the built-in slice viewers from `pygeoinf3d.plot` so you can inspect the KL samples with moving planar cuts through the volume.


In [ ]:
plot_function3d_axis_slider(
    samples_fn[0],
    axis='z',
    n_slices=30,
    resolution=80,
    colorscale='RdBu_r',
    title='KL Sample 1 — z-axis slice slider',
    plot_n_modes=8000,
    show_plot=False,
)

In [ ]:
compare_function3d_axis_sliders(
    [samples_fn[0], samples_fn[1]],
    axis='z',
    n_slices=30,
    resolution=80,
    titles=sample_labels[:2],
    colorscale='RdBu_r',
    plot_n_modes=500,
    show_plot=False,
)